In [68]:
# conda activate dream

library(edgeR)
library(limma)
library(data.table)

setwd("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/frontal_cortex")

## Regress out covariates starting with OR counts

In [ ]:
counts <- fread("GTEx_frontal_cortex_TPM_SampleNetworks/All_09-32-34/GTEx_frontal_cortex_TPM_All_186_outliers_removed.csv", data.table=FALSE)
meta <- fread("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/bulk/GTEx/frontal_cortex/GTEx_frontal_cortex_sampleinfo.csv", data.table=FALSE)

In [9]:
meta[,1] <- make.names(meta[,1])

In [10]:
meta <- meta[meta[,1] %in% colnames(counts),]
counts <- counts[, c(1, match(meta[,1], colnames(counts)))]

In [12]:
all.equal(meta[,1], colnames(counts)[-1])

[1] TRUE

In [26]:
prob <- .6
mean_expr <- rowMeans(counts[,-1])
print(sum(mean_expr >= quantile(mean_expr, prob)))
counts_filtered <- counts[mean_expr >= prob,]

[1] 22111


In [45]:
# 1) Normalize
dge <- DGEList(counts=counts_filtered)
dge <- calcNormFactors(dge) 
logCPM <- cpm(dge, log=FALSE, prior.count=1)

Setting first column of `counts` as gene annotation.



In [ ]:
# 2) Define covariates
meta$SMNABTCH <- factor(meta$SMNABTCH)
meta$SMGEBTCH <- factor(meta$SMGEBTCH)

# treat Hardy etc as factors if you include them later
# meta$DTHHRDY <- factor(meta$DTHHRDY)

# (optional) scale continuous covariates so coefficients behave nicely
scale_if_num <- function(x) if(is.numeric(x)) as.numeric(scale(x)) else x
meta$SMRIN    <- scale_if_num(meta$SMRIN)
meta$SMTSISCH <- scale_if_num(meta$SMTSISCH)

# Example QC covariates (see list below)
meta$SMMAPRT  <- scale_if_num(meta$SMMAPRT)
meta$SMEXNCRT <- scale_if_num(meta$SMEXNCRT)
meta$SMRRNART <- scale_if_num(meta$SMRRNART)
meta$SMDPMPRT <- scale_if_num(meta$SMDPMPRT)

In [41]:
length(unique(meta$SMNABTCH))

[1] 121

In [47]:
length(unique(meta$SMGEBTCH))

[1] 83

In [ ]:
# 3) Fit gene-wise linear model and take residuals (+ add mean back)

X <- model.matrix(~ SMNABTCH + SMRIN + SMTSISCH +
                    SMMAPRT + SMEXNCRT + SMRRNART + SMDPMPRT,
                  data=meta)

fit <- lmFit(logCPM, X)

In [ ]:
fitted_vals <- fit$coefficients %*% t(X)   # G x N predicted expression
resid_mat <- logCPM - fitted_vals 

In [ ]:
logCPM_adj <- resid_mat + rowMeans(logCPM)
expr <- data.frame(Gene=dge$genes[,1], logCPM_adj)
expr[1:5,1:5]

In [67]:
fwrite(expr, file="data/GTEx_frontal_cortex_TPM_OR_lmFit_residuals.csv")

## Regress out covariates starting with OR QN counts

In [91]:
counts <- fread("GTEx_frontal_cortex_TPM_OR_QN_SampleNetworks/All_09-52-57/GTEx_frontal_cortex_TPM_OR_QN_All_166_outliers_removed.csv", data.table=FALSE)
meta <- fread("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/bulk/GTEx/frontal_cortex/GTEx_frontal_cortex_sampleinfo.csv", data.table=FALSE)

In [92]:
meta[,1] <- make.names(meta[,1])

In [93]:
meta <- meta[meta[,1] %in% colnames(counts),]
counts <- counts[, c(1, match(meta[,1], colnames(counts)))]

In [94]:
all.equal(meta[,1], colnames(counts)[-1])

[1] TRUE

In [95]:
prob <- .6
mean_expr <- rowMeans(counts[,-1])
print(sum(mean_expr >= quantile(mean_expr, prob)))
counts_filtered <- counts[mean_expr >= prob,]

[1] 21748


In [96]:
# 1) Normalize
dge <- DGEList(counts=counts_filtered)
dge <- calcNormFactors(dge) 
logCPM <- cpm(dge, log=FALSE, prior.count=1)

Setting first column of `counts` as gene annotation.



In [ ]:
# 2) Define covariates
meta$SMNABTCH <- factor(meta$SMNABTCH)
meta$SMGEBTCH <- factor(meta$SMGEBTCH)

# treat Hardy etc as factors if you include them later
# meta$DTHHRDY <- factor(meta$DTHHRDY)

# (optional) scale continuous covariates so coefficients behave nicely
scale_if_num <- function(x) if(is.numeric(x)) as.numeric(scale(x)) else x
meta$SMRIN    <- scale_if_num(meta$SMRIN)
meta$SMTSISCH <- scale_if_num(meta$SMTSISCH)

# Example QC covariates (see list below)
meta$SMMAPRT  <- scale_if_num(meta$SMMAPRT) # Mapping Rate: Proportion of Mapped Reads to Total Reads
meta$SMEXNCRT <- scale_if_num(meta$SMEXNCRT) # Exonic Rate: Proportion of Exonic Reads among Mapped Reads
meta$SMRRNART <- scale_if_num(meta$SMRRNART) # rRNA Rate: Proportion of rRNA Reads among Mapped Reads
meta$SMDPMPRT <- scale_if_num(meta$SMDPMPRT) # Duplicate Rate of Mapped: Proportion of Mapped Duplicate Reads to Mapped Reads

In [98]:
length(unique(meta$SMNABTCH))

[1] 110

In [99]:
length(unique(meta$SMGEBTCH))

[1] 78

In [100]:
dim(meta)

[1] 166 124

In [101]:
# 3) Fit gene-wise linear model and take residuals (+ add mean back)

X <- model.matrix(~ SMNABTCH + SMRIN + SMTSISCH +
                    SMMAPRT + SMEXNCRT + SMRRNART + SMDPMPRT,
                  data=meta)

fit <- lmFit(logCPM, X)

In [102]:
fitted_vals <- fit$coefficients %*% t(X)   # G x N predicted expression
resid_mat <- logCPM - fitted_vals 

In [103]:
logCPM_adj <- resid_mat + rowMeans(logCPM)
expr <- data.frame(Gene=dge$genes[,1], logCPM_adj)
expr[1:5,1:5]

,Gene,GTEX.1117F.0011.R10b.SM.GI4VE,GTEX.111FC.0011.R10a.SM.GIN8G,GTEX.117XS.0011.R10b.SM.GIN8Z,GTEX.1192X.0011.R10a.SM.DO941
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
2,WASH7P,2.523525,2.356629,2.356629,2.356629
10,ENSG00000268903,3.225272,2.309973,2.309973,2.309973
11,ENSG00000269981,2.396081,2.514633,2.514633,2.514633
16,WASH9P,3.950376,4.592253,4.592253,4.592253
29,MTND1P23,113.805089,175.870139,175.870139,175.870139


In [104]:
fwrite(expr, file="data/GTEx_frontal_cortex_TPM_OR_QN_lmFit_residuals.csv")